# LeetCode 4111 — Count Consistent Subarrays I bằng Python

Link đề: https://leetcode.com/problems/count-consistent-subarrays-i/ · Độ khó: **Easy** · Chủ đề: Array, Bit Manipulation (AND theo mảng con)

Skeleton LeetCode:

```python
class Solution:
    def countConsistent(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề, ví dụ và constraints lấy từ bản mô tả gốc; mọi đáp án trong notebook đều được kiểm chứng bằng code và đối chiếu với một lời giải brute force `O(n^3)` viết độc lập (tính AND và min từ đầu cho từng mảng con).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một **mảng con** (subarray, đoạn liên tiếp, không rỗng) là **consistent** nếu **AND bit** của mọi phần tử bằng **giá trị nhỏ nhất** của nó; đếm số mảng con như vậy.
2. Nắm toán tử **`&` (bitwise AND)** và hai tính chất quan trọng: `a & b <= min(a, b)`, và AND của một đoạn chỉ có thể **giảm** khi thêm phần tử.
3. Đi từ cách **vét cạn `O(n^3)`** (tính lại AND/min cho từng đoạn) → **cập nhật dần khi mở rộng đầu phải `O(n^2)`** → **hiểu điều kiện bằng "tập con bit" (submask)** → **đếm theo phần tử nhỏ nhất (leftmost minimum)**.
4. Biết kỹ thuật **cố định đầu trái `i`, quét đầu phải `j`**, giữ hai biến chạy `x` (AND) và `y` (min).
5. Tránh các lỗi kinh điển: quên mảng con độ dài 1, không **reset** biến chạy khi đổi `i`, dùng OR thay AND, và **dừng sớm** (`break`) khi tạm thời `AND != min` (sai!).

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`.

Một **mảng con** (subarray) được gọi là **consistent** nếu **AND bit (bitwise AND)** của **tất cả** phần tử của nó **bằng giá trị nhỏ nhất** (minimum) của mảng con đó.

Trả về **số lượng mảng con consistent** trong `nums`.

**Ví dụ 1**

- Input: `nums = [5,3,1,2]`
- Output: `6`
- Giải thích: các mảng con consistent là `[5]`, `[3]`, `[1]`, `[2]` (mỗi phần tử đơn lẻ có AND = min = chính nó), `[3,1]` (AND `3 & 1 = 1`, min `1`) và `[5,3,1]` (AND `5 & 3 & 1 = 1`, min `1`). Tổng `6`.

**Ví dụ 2**

- Input: `nums = [1,1,1]`
- Output: `6`
- Giải thích: mọi mảng con (có `3 + 2 + 1 = 6` mảng) đều có AND = min = `1`.

**Ví dụ 3**

- Input: `nums = [0]`
- Output: `1`
- Giải thích: mảng con duy nhất là `[0]`, AND và min đều bằng `0`.

**Ràng buộc:** `1 <= nums.length <= 100`; `0 <= nums[i] <= 100`.

In [1]:
from typing import List
from functools import reduce
from operator import and_
import random

examples = [
    ([5, 3, 1, 2], 6),
    ([1, 1, 1], 6),
    ([0], 1),
]
for nums, expected in examples:
    print(f"nums={nums} -> kỳ vọng {expected}")

nums=[5, 3, 1, 2] -> kỳ vọng 6
nums=[1, 1, 1] -> kỳ vọng 6
nums=[0] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Mảng con (subarray) và số lượng mảng con

Mảng con là một **đoạn liên tiếp** `nums[i..j]` với `0 <= i <= j < n` (không bỏ chỗ nào, khác với subsequence). Có `n` mảng con bắt đầu ở mỗi vị trí giảm dần: tổng cộng `n(n+1)/2`. Với `[1, 1, 1]` là `3 + 2 + 1 = 6`.

In [2]:
nums = [5, 3, 1, 2]
n = len(nums)
total = 0
for i in range(n):
    for j in range(i, n):
        print(f"[{i}..{j}] = {nums[i:j + 1]}")
        total += 1
print("tổng số mảng con:", total, "= n(n+1)/2 =", n * (n + 1) // 2)

[0..0] = [5]
[0..1] = [5, 3]
[0..2] = [5, 3, 1]
[0..3] = [5, 3, 1, 2]
[1..1] = [3]
[1..2] = [3, 1]
[1..3] = [3, 1, 2]
[2..2] = [1]
[2..3] = [1, 2]
[3..3] = [2]
tổng số mảng con: 10 = n(n+1)/2 = 10


### 3.2. Toán tử AND bit `&`

`a & b` đặt bit kết quả bằng `1` **chỉ khi cả hai** bit tương ứng đều là `1`. Ví dụ `5 = 101`, `3 = 011`, nên `5 & 3 = 001 = 1`.

| bit của a | bit của b | `a & b` |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 0 |
| 1 | 0 | 0 |
| 1 | 1 | 1 |

In [3]:
a, b = 5, 3
print(f"{a:03b} & {b:03b} = {a & b:03b} = {a & b}")
print(f"{3:03b} & {1:03b} = {3 & 1:03b}")
print(5 & 3 & 1)             # AND nhiều số: kết hợp từ trái sang phải
print(7 & 7, 6 & 0)          # x & x = x ; x & 0 = 0

101 & 011 = 001 = 1
011 & 001 = 001
1
7 0


### 3.3. AND nhiều số và `&=`

AND có tính **giao hoán** và **kết hợp**, nên AND của cả đoạn tính dần được: `x = nums[i]`, rồi `x &= nums[j]` với mỗi `j`. Viết `x &= v` tương đương `x = x & v`. Cũng có thể gộp cả danh sách bằng `reduce`.

In [4]:
nums = [5, 3, 1]
x = nums[0]
for v in nums[1:]:
    x &= v
    print("x =", x)
print(reduce(and_, nums))        # 1

x = 1
x = 1
1


### 3.4. Tính chất 1: AND chỉ có thể **giảm** (hoặc giữ nguyên) khi thêm phần tử

Vì `x & v` chỉ **tắt** bit chứ không bật thêm bit, nên `x & v <= x`. Do đó, AND của `nums[i..j]` không bao giờ tăng khi `j` tăng. Tương tự **min** của đoạn cũng chỉ giảm khi thêm phần tử. Hai đại lượng cùng giảm, nhưng **không** cùng tốc độ: đó là chỗ thú vị của bài.

In [5]:
nums = [5, 3, 1, 2]
x = nums[0]
y = nums[0]
for j, v in enumerate(nums):
    x &= v
    y = min(y, v)
    print(f"j={j}: AND={x}, min={y}")

j=0: AND=5, min=5
j=1: AND=1, min=3
j=2: AND=1, min=1
j=3: AND=0, min=1


### 3.5. Tính chất 2: `AND <= min` luôn đúng

Gọi `m` là phần tử nhỏ nhất của đoạn. AND của cả đoạn là AND của `m` với các số khác, nên `AND <= m` (AND chỉ tắt bit của `m`). Vậy **`AND == min` xảy ra khi và chỉ khi không bit nào của `m` bị tắt**, tức **mọi phần tử của đoạn đều chứa mọi bit của `m`** (`v & m == m` với mọi `v` trong đoạn). Nói cách khác, `m` là một **tập con bit (submask)** của mọi phần tử.

In [6]:
import itertools
nums = [5, 3, 1, 2]
for i, j in [(0, 1), (1, 2), (0, 2)]:
    seg = nums[i:j + 1]
    m = min(seg)
    all_super = all(v & m == m for v in seg)
    print(seg, "AND =", reduce(and_, seg), "min =", m, "| mọi phần tử chứa bit của min?", all_super)

[5, 3] AND = 1 min = 3 | mọi phần tử chứa bit của min? False
[3, 1] AND = 1 min = 1 | mọi phần tử chứa bit của min? True
[5, 3, 1] AND = 1 min = 1 | mọi phần tử chứa bit của min? True


### 3.6. `min`, `enumerate`, `range(i, n)`

- `min(a, b)` trả số nhỏ hơn; `min(seq)` trả nhỏ nhất.
- `for i, x in enumerate(nums)` cho chỉ số và giá trị cùng lúc.
- `range(i, n)` sinh `i, i+1, ..., n-1`: các đầu phải hợp lệ của đoạn bắt đầu ở `i` (đầu phải **bằng** `i` là mảng con độ dài 1).

In [7]:
nums = [5, 3, 1, 2]
for i, x in enumerate(nums):
    print(f"i={i}, x={x}, các đầu phải j:", list(range(i, len(nums))))

i=0, x=5, các đầu phải j: [0, 1, 2, 3]
i=1, x=3, các đầu phải j: [1, 2, 3]
i=2, x=1, các đầu phải j: [2, 3]
i=3, x=2, các đầu phải j: [3]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Tính tay ví dụ 1, `nums = [5, 3, 1, 2]` (`5 = 101`, `3 = 011`, `1 = 001`, `2 = 010`):

| đoạn | AND | min | consistent? |
|---|---|---|---|
| `[5]` | 5 | 5 | có |
| `[5,3]` | `101 & 011 = 001 = 1` | 3 | không (1 ≠ 3) |
| `[5,3,1]` | 1 | 1 | **có** |
| `[5,3,1,2]` | `001 & 010 = 0` | 1 | không (0 ≠ 1) |
| `[3]` | 3 | 3 | có |
| `[3,1]` | 1 | 1 | có |
| `[3,1,2]` | 0 | 1 | không |
| `[1]` | 1 | 1 | có |
| `[1,2]` | 0 | 1 | không |
| `[2]` | 2 | 2 | có |

Đếm các dòng "có": `[5]`, `[5,3,1]`, `[3]`, `[3,1]`, `[1]`, `[2]` = **6**. Điều đáng chú ý: `[5,3]` **không** consistent nhưng `[5,3,1]` **lại có** — nên không thể "dừng" khi gặp một đoạn không đạt.

## 5. Cách 1 — Brute Force / baseline: tính lại AND và min cho từng đoạn (`O(n^3)`)

**Ý tưởng.** Với mọi cặp `(i, j)`, cắt `seg = nums[i:j+1]`, tính AND bằng `reduce`/vòng lặp và min bằng `min(seg)`, rồi so sánh.

In [8]:
def count_cubic(nums):
    n = len(nums)
    count = 0
    for i in range(n):
        for j in range(i, n):
            seg = nums[i:j + 1]                 # đoạn đang xét
            and_all = seg[0]
            for v in seg[1:]:
                and_all &= v                    # AND tính lại từ đầu
            if and_all == min(seg):             # min cũng tính lại từ đầu
                count += 1
    return count


for nums, expected in examples:
    print(nums, "->", count_cubic(nums), "(kỳ vọng", expected, ")")

[5, 3, 1, 2] -> 6 (kỳ vọng 6 )
[1, 1, 1] -> 6 (kỳ vọng 6 )
[0] -> 1 (kỳ vọng 1 )


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`. Có `O(n^2)` đoạn, mỗi đoạn tốn `O(n)` để AND và min: **Time `O(n^3)`**, **Space `O(n)`** do lát cắt `seg`. Với `n <= 100` thì khoảng `~170 000` phép — vẫn chạy được, nhưng lãng phí.

**Nút thắt.** Khi `j` tăng thêm `1`, đoạn mới chỉ thêm đúng `nums[j]`; AND mới và min mới đều suy ra từ AND cũ, min cũ trong `O(1)`. Việc tính lại từ đầu là thừa.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát 1 (cập nhật dần):** nếu `x` là AND của `nums[i..j-1]` và `y` là min của nó, thì AND của `nums[i..j]` là `x & nums[j]` và min là `min(y, nums[j])`. Mỗi bước `O(1)`.

**Quan sát 2 (cố định đầu trái):** với mỗi `i`, đặt `x = y = nums[i]` rồi cho `j` chạy từ `i` tới `n-1` (đầu phải), đếm mỗi lần `x == y`. Nhớ **reset** `x, y` khi đổi `i`.

**Quan sát 3 (điều kiện submask):** vì `AND <= min`, đoạn consistent chính là đoạn mà **mọi phần tử chứa mọi bit của min**. Điều này cho một cách đếm khác: với mỗi phần tử `v = nums[m]` đóng vai "min", ta mở rộng sang hai phía qua các phần tử `x` thỏa `x & v == v` (nghĩa là `x` chứa bit của `v`, nên `x >= v`) — tương tự bài "đếm đoạn mà `nums[m]` là min".

Kết quả: **`O(n^2)` thời gian, `O(1)` bộ nhớ** là đủ rộng rãi với `n <= 100`.

## 7. Các cách tiếp theo

### Cách 2 — Cố định đầu trái, quét đầu phải, cập nhật `x` (AND) và `y` (min)

**Giải thích.** Hai vòng `for` lồng nhau; `x` và `y` khởi tạo bằng `nums[i]`, mỗi bước `j` cập nhật rồi so sánh.

In [9]:
def count_quadratic(nums):
    n = len(nums)
    count = 0
    for i in range(n):
        x = y = nums[i]                   # reset cho mỗi đầu trái i
        for j in range(i, n):
            x &= nums[j]                  # AND của nums[i..j]
            y = min(y, nums[j])           # min của nums[i..j]
            if x == y:
                count += 1
    return count


for nums, expected in examples:
    print(nums, "->", count_quadratic(nums), "(kỳ vọng", expected, ")")

[5, 3, 1, 2] -> 6 (kỳ vọng 6 )
[1, 1, 1] -> 6 (kỳ vọng 6 )
[0] -> 1 (kỳ vọng 1 )


In [10]:
# Code nộp LeetCode cho Cách 2
class Solution:
    def countConsistent(self, nums: List[int]) -> int:
        n = len(nums)
        ans = 0
        for i in range(n):
            x = y = nums[i]
            for j in range(i, n):
                x &= nums[j]
                y = min(y, nums[j])
                if x == y:
                    ans += 1
        return ans


print(Solution().countConsistent([5, 3, 1, 2]))   # 6

6


**Độ phức tạp Cách 2:** **Time `O(n^2)`** (`n(n+1)/2` bước, mỗi bước `O(1)`), **Space `O(1)`**.

### Cách 3 — Đếm theo "phần tử nhỏ nhất ở vị trí trái nhất" (submask mở rộng hai phía)

**Giải thích.** Mỗi mảng con consistent có min `v`; ta gán nó cho **vị trí trái nhất** `m` có `nums[m] == v` trong mảng con. Với `m` cố định, đoạn hợp lệ có đầu trái `l` và đầu phải `r` thỏa:

- mọi phần tử bên **trái** `m` trong đoạn `(l..m-1)` phải **chứa bit của `v` và khác `v`** (`x & v == v` và `x != v`, tức `x > v` để `m` là vị trí trái nhất của min);
- mọi phần tử bên **phải** `m` trong đoạn `(m+1..r)` phải **chứa bit của `v`** (`x & v == v`, có thể bằng `v`).

Số lựa chọn bên trái = `(độ dài dãy liên tiếp bên trái thỏa) + 1`, bên phải tương tự; số đoạn gán cho `m` bằng **tích hai số đó**.

In [11]:
def count_by_minimum(nums):
    n = len(nums)
    total = 0
    for m in range(n):
        v = nums[m]
        left = 0
        k = m - 1
        while k >= 0 and nums[k] & v == v and nums[k] != v:     # bên trái: chứa bit của v và lớn hơn v
            left += 1
            k -= 1
        right = 0
        k = m + 1
        while k < n and nums[k] & v == v:                        # bên phải: chứa bit của v (có thể bằng v)
            right += 1
            k += 1
        total += (left + 1) * (right + 1)
    return total


for nums, expected in examples:
    print(nums, "->", count_by_minimum(nums), "(kỳ vọng", expected, ")")

[5, 3, 1, 2] -> 6 (kỳ vọng 6 )
[1, 1, 1] -> 6 (kỳ vọng 6 )
[0] -> 1 (kỳ vọng 1 )


**Độ phức tạp Cách 3:** **Time `O(n^2)` trong worst case** (với mỗi `m` ta có thể mở rộng gần hết mảng), nhưng thường ít hơn rất nhiều khi dữ liệu ngẫu nhiên; **Space `O(1)`**. Cách này cho thấy bài **không phải** chỉ là "thử mọi đoạn": điều kiện submask cho phép đếm theo cấu trúc.

> Cách 2 là bản nộp rõ ràng, an toàn nhất: ngắn, `O(n^2)` với `n <= 100`.

### Cách 4 — `itertools.accumulate` cho AND và min chạy

**Giải thích.** Với mỗi `i`, `accumulate(nums[i:], and_)` cho các AND chạy, `accumulate(nums[i:], min)` cho các min chạy; `zip` hai dãy và đếm số cặp bằng nhau. Code ngắn, nhưng tạo lát cắt nên tốn thêm bộ nhớ.

In [12]:
from itertools import accumulate


def count_accumulate(nums):
    count = 0
    for i in range(len(nums)):
        tail = nums[i:]
        count += sum(a == b for a, b in zip(accumulate(tail, and_), accumulate(tail, min)))
    return count


for nums, expected in examples:
    print(nums, "->", count_accumulate(nums), "(kỳ vọng", expected, ")")

[5, 3, 1, 2] -> 6 (kỳ vọng 6 )
[1, 1, 1] -> 6 (kỳ vọng 6 )
[0] -> 1 (kỳ vọng 1 )


**Độ phức tạp Cách 4:** **Time `O(n^2)`**, **Space `O(n)`** (lát cắt `tail`).

## 8. TRACE từng bước

Ta in toàn bộ quá trình của Cách 2 cho `nums = [5, 3, 1, 2]`: với mỗi `(i, j)`, in `x` (AND), `y` (min) và quyết định đếm.

In [13]:
def trace_quadratic(nums):
    print("nums =", nums)
    n = len(nums)
    count = 0
    for i in range(n):
        x = y = nums[i]
        print(f"--- đầu trái i={i}, reset x=y={nums[i]}")
        for j in range(i, n):
            x &= nums[j]
            y = min(y, nums[j])
            ok = x == y
            count += ok
            print(f"  j={j}: nums[{i}..{j}]={nums[i:j+1]!s:<12} AND={x} min={y} -> {'đếm' if ok else 'bỏ'} (count={count})")
    print("kết quả:", count)
    return count


trace_quadratic([5, 3, 1, 2])

nums = [5, 3, 1, 2]
--- đầu trái i=0, reset x=y=5
  j=0: nums[0..0]=[5]          AND=5 min=5 -> đếm (count=1)
  j=1: nums[0..1]=[5, 3]       AND=1 min=3 -> bỏ (count=1)
  j=2: nums[0..2]=[5, 3, 1]    AND=1 min=1 -> đếm (count=2)
  j=3: nums[0..3]=[5, 3, 1, 2] AND=0 min=1 -> bỏ (count=2)
--- đầu trái i=1, reset x=y=3
  j=1: nums[1..1]=[3]          AND=3 min=3 -> đếm (count=3)
  j=2: nums[1..2]=[3, 1]       AND=1 min=1 -> đếm (count=4)
  j=3: nums[1..3]=[3, 1, 2]    AND=0 min=1 -> bỏ (count=4)
--- đầu trái i=2, reset x=y=1
  j=2: nums[2..2]=[1]          AND=1 min=1 -> đếm (count=5)
  j=3: nums[2..3]=[1, 2]       AND=0 min=1 -> bỏ (count=5)
--- đầu trái i=3, reset x=y=2
  j=3: nums[3..3]=[2]          AND=2 min=2 -> đếm (count=6)
kết quả: 6


6

Bảng tóm tắt các lần đếm (ví dụ 1):

| đầu trái `i` | các `j` được đếm | số đoạn |
|---|---|---|
| 0 | `j=0` (`[5]`), `j=2` (`[5,3,1]`) | 2 |
| 1 | `j=1` (`[3]`), `j=2` (`[3,1]`) | 2 |
| 2 | `j=2` (`[1]`) | 1 |
| 3 | `j=3` (`[2]`) | 1 |

Tổng `2 + 2 + 1 + 1 = 6`. Hãy nhìn riêng `i = 0`: `j=1` **không** đếm (`AND = 1`, `min = 3`) nhưng `j=2` **lại đếm** (`AND = 1`, `min = 1`). Đó là lý do ta không được `break`.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (hai vòng lặp, cập nhật dần). Đúng `class Solution`, đúng tên method `countConsistent`.

In [14]:
class Solution:
    def countConsistent(self, nums: List[int]) -> int:
        n = len(nums)
        ans = 0
        for i in range(n):
            x = y = nums[i]               # AND và min của đoạn nums[i..i]
            for j in range(i, n):
                x &= nums[j]
                y = min(y, nums[j])
                if x == y:
                    ans += 1
        return ans


sol = Solution()
for nums, expected in examples:
    assert sol.countConsistent(nums) == expected
print("OK: 3 ví dụ của đề")

OK: 3 ví dụ của đề


## 10. Vì sao thuật toán đúng

- **Phủ hết, không trùng:** cặp `(i, j)` với `0 <= i <= j < n` xác định đúng một mảng con và mọi mảng con đều có mặt một lần.
- **Cập nhật dần đúng:** AND và min là các phép **kết hợp**: `AND(nums[i..j]) = AND(nums[i..j-1]) & nums[j]` và `min(nums[i..j]) = min(min(nums[i..j-1]), nums[j])`. Khởi tạo `x = y = nums[i]` rồi `j = i` vẫn đúng vì `v & v = v` và `min(v, v) = v` (mảng con độ dài 1 luôn consistent).
- **Reset:** `x`, `y` được gán lại ở **mỗi** `i`, nên không rò rỉ trạng thái giữa các đầu trái.
- **Không được dừng sớm:** `AND` và `min` đều giảm nhưng không đồng bộ; ví dụ `[5,3,1]`: tại `j=1` `AND (1) < min (3)` (không đạt) nhưng tại `j=2` min giảm xuống `1` đuổi kịp AND. Vì vậy phải duyệt hết `j`.
- **Cách 3 đúng:** vì `AND <= min`, đoạn consistent ⟺ mọi phần tử chứa bit của min; gán đoạn cho vị trí trái nhất của min giúp **đếm mỗi đoạn đúng một lần**.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (tối đa `100`).

- **Cách 1 (tính lại):** Time `O(n^3)`, Space `O(n)`.
- **Cách 2 (cập nhật dần):** Time `O(n^2)`, Space `O(1)` — **bản nộp**; `n(n+1)/2 <= 5050` bước.
- **Cách 3 (theo phần tử min):** Time `O(n^2)` worst case (ví dụ `nums` toàn các số bằng nhau thì mỗi `m` mở rộng gần hết bên phải), thường nhanh hơn; Space `O(1)`.
- **Cách 4 (`accumulate`):** Time `O(n^2)`, Space `O(n)`.

Thuật toán không phụ thuộc nhiều vào dữ liệu ở Cách 1/2/4 (luôn `n(n+1)/2` hoặc `~n^3/6` bước); chỉ Cách 3 có khác biệt average/worst.

## 12. Bộ test

`oracle` là bản brute force độc lập: kiểm tra **trực tiếp định nghĩa** bằng cách duyệt từng chỉ số của đoạn (không dùng `reduce`, không dùng cập nhật dần). Đối chiếu mọi cách.

In [15]:
def oracle(nums):
    """Brute force độc lập: với mỗi đoạn, tính AND bằng duyệt từng bit 0..6 và min bằng duyệt."""
    n = len(nums)
    ans = 0
    for i in range(n):
        for j in range(i, n):
            bits = 0
            for b in range(7):                       # 100 < 128 nên đủ 7 bit
                if all((nums[k] >> b) & 1 for k in range(i, j + 1)):
                    bits |= 1 << b                   # bit b bằng 1 chỉ khi MỌI phần tử có bit b
            smallest = nums[i]
            for k in range(i, j + 1):
                if nums[k] < smallest:
                    smallest = nums[k]
            if bits == smallest:
                ans += 1
    return ans


impls = {
    "cubic": count_cubic,
    "quadratic": count_quadratic,
    "by_minimum": count_by_minimum,
    "accumulate": count_accumulate,
    "submit": sol.countConsistent,
}

for nums, expected in examples:
    assert oracle(nums) == expected
    for key, fn in impls.items():
        assert fn(nums) == expected, (key, nums, fn(nums), expected)
print("PASS ví dụ của đề (mọi cách và oracle)")

PASS ví dụ của đề (mọi cách và oracle)


### 12.1. Các trường hợp biên

In [16]:
edge_cases = [
    ([0], 1, "một phần tử 0"),
    ([7], 1, "một phần tử"),
    ([0, 0, 0], 6, "toàn 0: mọi đoạn đạt"),
    ([3, 4], 2, "3 & 4 = 0, min = 3: chỉ hai đoạn đơn"),
    ([4, 6], 3, "4 & 6 = 4 = min: cả [4], [6], [4,6]"),
    ([6, 4], 3, "đảo thứ tự vẫn như vậy"),
    ([5, 3, 1], 5, "[5,3] không đạt nhưng [5,3,1] đạt: [5],[3],[1],[3,1],[5,3,1]"),
    ([1, 2, 4, 8], 4, "các lũy thừa của 2 rời nhau: chỉ đoạn đơn"),
    ([0, 5, 0], 6, "mọi đoạn chứa số 0 có AND = 0 = min nên đều đạt; chỉ [5] đạt riêng: tổng 6"),
]
for nums, expected, label in edge_cases:
    assert oracle(nums) == expected, (label, oracle(nums), expected)
    for key, fn in impls.items():
        assert fn(nums) == expected, (key, label, nums, fn(nums), expected)
    print("PASS", label)

PASS một phần tử 0
PASS một phần tử
PASS toàn 0: mọi đoạn đạt
PASS 3 & 4 = 0, min = 3: chỉ hai đoạn đơn
PASS 4 & 6 = 4 = min: cả [4], [6], [4,6]
PASS đảo thứ tự vẫn như vậy
PASS [5,3] không đạt nhưng [5,3,1] đạt: [5],[3],[1],[3,1],[5,3,1]
PASS các lũy thừa của 2 rời nhau: chỉ đoạn đơn
PASS mọi đoạn chứa số 0 có AND = 0 = min nên đều đạt; chỉ [5] đạt riêng: tổng 6


### 12.2. Tính chất và phản ví dụ

- **Mọi đoạn độ dài 1 đều consistent**, nên đáp án `>= n`; và `<= n(n+1)/2`.
- Mảng **toàn bằng nhau** cho đúng `n(n+1)/2`.
- **Phản ví dụ** cho lỗi `break` sớm: `[5, 3, 1]` đúng là `5`, nhưng nếu dừng khi `AND != min` thì ta mất đoạn `[5,3,1]` và trả `4`.

In [17]:
random.seed(111)
for _ in range(400):
    n = random.randint(1, 15)
    nums = [random.randint(0, 100) for _ in range(n)]
    ans = sol.countConsistent(nums)
    assert n <= ans <= n * (n + 1) // 2
for n in range(1, 12):
    for v in (0, 1, 5, 100):
        assert sol.countConsistent([v] * n) == n * (n + 1) // 2
print("PASS tính chất (cận n..n(n+1)/2, mảng đều)")


def sai_break_som(nums):
    n = len(nums)
    ans = 0
    for i in range(n):
        x = y = nums[i]
        for j in range(i, n):
            x &= nums[j]
            y = min(y, nums[j])
            if x == y:
                ans += 1
            else:
                break                                   # SAI: dừng sớm
    return ans


print("phản ví dụ [5, 3, 1]: đúng =", sol.countConsistent([5, 3, 1]), "| break sớm =", sai_break_som([5, 3, 1]))
assert sai_break_som([5, 3, 1]) != sol.countConsistent([5, 3, 1])

PASS tính chất (cận n..n(n+1)/2, mảng đều)
phản ví dụ [5, 3, 1]: đúng = 5 | break sớm = 4


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [18]:
random.seed(4111)
for trial in range(1500):
    n = random.randint(1, 14)
    hi = random.choice([1, 3, 7, 15, 100])
    nums = [random.randint(0, hi) for _ in range(n)]
    truth = oracle(nums)
    for key, fn in impls.items():
        assert fn(nums) == truth, (key, nums, fn(nums), truth)
print("PASS 1500 mảng ngẫu nhiên nhỏ (nhiều giá trị trùng, nhiều số 0): mọi cách khớp oracle")

# Mảng dài tới 100 phần tử: so các cách nhanh với nhau và với cách O(n^3)
for trial in range(60):
    n = random.randint(60, 100)
    hi = random.choice([3, 15, 100])
    nums = [random.randint(0, hi) for _ in range(n)]
    ref = count_cubic(nums)
    for key, fn in impls.items():
        assert fn(nums) == ref, (key, n, fn(nums), ref)
print("PASS 60 mảng dài (n tới 100): mọi cách khớp cách O(n^3)")

PASS 1500 mảng ngẫu nhiên nhỏ (nhiều giá trị trùng, nhiều số 0): mọi cách khớp oracle


PASS 60 mảng dài (n tới 100): mọi cách khớp cách O(n^3)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Không reset `x`, `y` khi đổi đầu trái `i`
#### Code sai

In [19]:
def sai_khong_reset(nums):
    n = len(nums)
    ans = 0
    x = y = nums[0]                     # SAI: đặt ngoài vòng lặp i
    for i in range(n):
        for j in range(i, n):
            x &= nums[j]
            y = min(y, nums[j])
            if x == y:
                ans += 1
    return ans


print("đúng:", Solution().countConsistent([5, 3, 1, 2]), "| sai:", sai_khong_reset([5, 3, 1, 2]))

đúng: 6 | sai: 2


**Vì sao sai:** AND và min của đoạn mới (đầu trái `i`) bị lẫn trạng thái của các đoạn trước, có thể đã bị kéo xuống rất thấp. **Code đúng:** đặt `x = y = nums[i]` **bên trong** vòng `for i`.

### Lỗi 2: `break` sớm khi `AND != min`
#### Code sai
Xem hàm `sai_break_som` ở phần test.
#### Vì sao sai
AND và min đều chỉ giảm, nhưng có thể AND giảm trước rồi min giảm đuổi kịp (ví dụ `[5, 3, 1]`). Không có tính đơn điệu "đã sai thì sai mãi", nên phải duyệt hết đầu phải.

### Lỗi 3: Dùng OR (`|`) thay vì AND (`&`)
#### Code sai

In [20]:
def sai_dung_or(nums):
    n = len(nums)
    ans = 0
    for i in range(n):
        x = y = nums[i]
        for j in range(i, n):
            x |= nums[j]                 # SAI: OR
            y = min(y, nums[j])
            if x == y:
                ans += 1
    return ans


print("đúng:", Solution().countConsistent([5, 3, 1, 2]), "| sai (OR):", sai_dung_or([5, 3, 1, 2]))

đúng: 6 | sai (OR): 4


**Vì sao sai:** OR chỉ **tăng** (bật thêm bit), nên `OR >= max >= min` và chỉ bằng `min` khi mọi phần tử bằng nhau. Đề nói **AND**.

### Lỗi 4: Bắt đầu `j` từ `i + 1` (bỏ mảng con độ dài 1)
#### Code sai

In [21]:
def sai_bo_do_dai_1(nums):
    n = len(nums)
    ans = 0
    for i in range(n):
        x = y = nums[i]
        for j in range(i + 1, n):         # SAI: bỏ đoạn [i..i]
            x &= nums[j]
            y = min(y, nums[j])
            if x == y:
                ans += 1
    return ans


print("đúng:", Solution().countConsistent([5, 3, 1, 2]), "| sai:", sai_bo_do_dai_1([5, 3, 1, 2]))

đúng: 6 | sai: 2


**Vì sao sai:** mảng con độ dài 1 luôn thỏa (AND = min = chính nó) và đề tính chúng (ví dụ 3: `[0]` cho `1`). **Code đúng:** `for j in range(i, n)`.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "đếm số **mảng con** thỏa điều kiện" với `n <= 100` → thử ngay **hai vòng lặp `(i, j)`** và cập nhật đại lượng của đoạn dần dần (`O(n^2)`).
- Thấy **AND/OR/GCD/min/max/sum** của đoạn → các phép này **kết hợp được**, cập nhật `O(1)` khi mở rộng đầu phải; nhớ **reset khi đổi đầu trái**.
- Thấy **AND**: nhớ hai tính chất — AND chỉ **giảm** khi thêm phần tử; `AND <= min`. Điều kiện `AND == min` đồng nghĩa với "mọi phần tử chứa bit của min".
- **Không** vội `break`: chỉ `break` khi đại lượng có tính **đơn điệu** thật sự (ví dụ "AND bằng `0` rồi thì mãi bằng `0`").
- Đề tăng `n` lên `10^5` thì cần ý tưởng khác (phần tử min + mở rộng bằng **stack đơn điệu**, hoặc đối với AND thì mỗi `i` chỉ có `O(log max)` giá trị AND khác nhau).
- Luôn thử `[x]` (độ dài 1), mảng toàn bằng nhau và ví dụ có `0`.

## 15. Biến thể của bài

1. **Đếm đoạn có AND bằng `k`** cho trước (LeetCode 2444 / 2419 liên quan).
2. **Đếm đoạn có AND bằng max** (khi đó mọi phần tử phải bằng nhau): phát biểu và chứng minh.
3. **Count Consistent Subarrays II** (phiên bản `n` lớn hơn): cần thuật toán gần tuyến tính, dùng "các giá trị AND khác nhau chỉ `O(log max)`" hoặc stack đơn điệu.
4. Trả về **độ dài lớn nhất** của một mảng con consistent.
5. Đếm đoạn có `AND == min` và độ dài **chẵn**.

In [22]:
def longest_consistent(nums):
    n = len(nums)
    best = 0
    for i in range(n):
        x = y = nums[i]
        for j in range(i, n):
            x &= nums[j]
            y = min(y, nums[j])
            if x == y:
                best = max(best, j - i + 1)
    return best


def count_and_equals_max(nums):
    n = len(nums)
    ans = 0
    for i in range(n):
        x = y = nums[i]
        for j in range(i, n):
            x &= nums[j]
            y = max(y, nums[j])
            ans += x == y
    return ans


print(longest_consistent([5, 3, 1, 2]))        # 3: đoạn [5,3,1]
print(count_and_equals_max([5, 3, 1, 2]))      # 4: chỉ các đoạn đơn (mọi phần tử phải bằng nhau)
print(count_and_equals_max([2, 2, 2]))         # 6

3
4
6


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tính lại từng đoạn | cắt đoạn, AND và min từ đầu | `O(n^3)` | `O(n)` | kiểm tra đúng-sai, `n` rất nhỏ |
| 2. Cập nhật dần | cố định `i`, quét `j`, giữ `x` (AND), `y` (min) | `O(n^2)` | `O(1)` | **cách nộp chuẩn** với `n <= 100` |
| 3. Theo phần tử min | mở rộng hai phía qua các số chứa bit của min | `O(n^2)` worst | `O(1)` | mở đường cho bản `n` lớn (II) |
| 4. `accumulate` | `accumulate(and_)` và `accumulate(min)` rồi `zip` | `O(n^2)` | `O(n)` | code ngắn, đọc như "toán" |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `6 & 3`; (b) `5 & 5 & 0`; (c) `reduce(and_, [7, 6, 4])`; (d) `Solution().countConsistent([2, 2])`.

**Bài 2 (tính bằng tay).** Liệt kê mọi mảng con của `[4, 6, 5]`, tính AND và min, và đếm số mảng con consistent.

**Bài 3 (điền TODO).** Hoàn thiện `count_todo(nums)` (Cách 2).

In [23]:
def count_todo(nums):
    n = len(nums)
    ans = 0
    # TODO 1: duyệt i từ 0 tới n - 1
    # TODO 2: đặt x = y = nums[i] (reset cho mỗi i)
    # TODO 3: duyệt j từ i tới n - 1: x &= nums[j]; y = min(y, nums[j])
    # TODO 4: nếu x == y thì ans += 1
    # TODO 5: trả ans
    pass


# Tự làm xong rồi mới mở test:
# for nums, expected in examples:
#     assert count_todo(nums) == expected

**Bài 4 (tự viết lại).** Viết `count_submask(nums)` kiểm tra điều kiện bằng **submask**: với mỗi đoạn, `m = min(đoạn)` và đạt nếu `all(v & m == m for v in đoạn)`. So sánh với `Solution().countConsistent` trên mảng ngẫu nhiên.

**Bài 5 (biến thể / mini challenge).** Viết `count_and_equals_k(nums, k)` đếm số mảng con có AND bằng đúng `k`. Thử: `count_and_equals_k([5, 3, 1, 2], 1) == 4` (các đoạn `[5,3]`, `[5,3,1]`, `[3,1]`, `[1]`; hãy tự đếm tay rồi đối chiếu với code).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `&` chỉ giữ bit có ở **cả hai** số. Bài 2: có `3 + 2 + 1 = 6` mảng con. Bài 3: nhớ reset `x`, `y` mỗi `i`. Bài 5: chỉ thay điều kiện `x == y` thành `x == k`.
- **Mức 2 (vừa):** Bài 1: `6 & 3 = 2` (`110 & 011 = 010`); `0`; `7 & 6 & 4 = 4`; `3`. Bài 2: đoạn đơn `[4]`, `[6]`, `[5]` đều đạt; `[4,6]`: AND `4`, min `4` đạt; `[6,5]`: AND `4`, min `5` không; `[4,6,5]`: AND `4`, min `4` đạt. Bài 4: dùng lát cắt `nums[i:j+1]`.
- **Mức 3 (sâu):** Bài 2: các đoạn đạt là `[4]`, `[6]`, `[5]`, `[4,6]`, `[4,6,5]` = **5** (chỉ `[6,5]` không đạt). Bài 3: xem Cách 2. Bài 5: `x == k` thay cho `x == y`; với `[5,3,1,2]`, `k=1`: AND bằng `1` ở `[5,3]`, `[5,3,1]`, `[3,1]`, `[1]` (4 đoạn); các đoạn còn lại có AND là `5`, `3`, `0` hoặc `2`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [24]:
# Bài 1
assert 6 & 3 == 2 and 5 & 5 & 0 == 0 and reduce(and_, [7, 6, 4]) == 4
assert Solution().countConsistent([2, 2]) == 3

# Bài 2
assert oracle([4, 6, 5]) == 5 and Solution().countConsistent([4, 6, 5]) == 5


# Bài 3
def count_todo_ans(nums):
    n = len(nums)
    ans = 0
    for i in range(n):
        x = y = nums[i]
        for j in range(i, n):
            x &= nums[j]
            y = min(y, nums[j])
            if x == y:
                ans += 1
    return ans


for nums, expected in examples:
    assert count_todo_ans(nums) == expected


# Bài 4
def count_submask(nums):
    n = len(nums)
    ans = 0
    for i in range(n):
        for j in range(i, n):
            seg = nums[i:j + 1]
            m = min(seg)
            if all(v & m == m for v in seg):
                ans += 1
    return ans


random.seed(7)
for _ in range(300):
    arr = [random.randint(0, 20) for _ in range(random.randint(1, 12))]
    assert count_submask(arr) == Solution().countConsistent(arr)


# Bài 5
def count_and_equals_k(nums, k):
    n = len(nums)
    ans = 0
    for i in range(n):
        x = nums[i]
        for j in range(i, n):
            x &= nums[j]
            if x == k:
                ans += 1
    return ans


# [5,3,1,2], k=1: [1], [3,1], [5,3] (5&3=1), [5,3,1] -> 4
assert count_and_equals_k([5, 3, 1, 2], 1) == 4
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cố định đầu trái, quét đầu phải, cập nhật AND và min)

```python
class Solution:
    def countConsistent(self, nums: List[int]) -> int:
        n = len(nums)
        ans = 0
        for i in range(n):
            x = y = nums[i]
            for j in range(i, n):
                x &= nums[j]
                y = min(y, nums[j])
                if x == y:
                    ans += 1
        return ans
```

**Checklist**

- [ ] `x`, `y` được **reset** `= nums[i]` ở mỗi đầu trái `i`?
- [ ] `j` chạy từ `i` (có mảng con độ dài 1), không từ `i + 1`?
- [ ] Dùng **AND** `&` (không phải OR `|`) và **min**?
- [ ] **Không `break`** khi tạm thời `AND != min`?
- [ ] Đã thử `[0]`, mảng toàn bằng nhau, và `[5, 3, 1]` (đoạn `[5,3]` không đạt nhưng `[5,3,1]` đạt)?
- [ ] Đã đối chiếu với brute force tính lại từ đầu trên mảng ngẫu nhiên?
- [ ] Skeleton đúng `class Solution` / `def countConsistent(self, nums: List[int]) -> int`?